<a href="https://colab.research.google.com/github/K415mm/mcp_course_workshops/blob/main/Workshop_01_CTI_Automation/01_CTI_Automation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛡️ Workshop 1: CTI Automation with MCP

Welcome to the first official workshop of the RAISEGUARD Advanced Agentic AI Course!

**Goal:** Build a working MCP-powered Cyber Threat Intelligence (CTI) enrichment workflow that ingests IOCs from a raw alert, enriches each indicator using multiple sources, and produces a structured intelligence brief.

### Setup & Libraries
Run the cell below to install our dependencies.

In [ ]:
!pip install groq langchain-groq mcp requests python-whois -q
print("✅ Libraries installed successfully!")

### API Keys
For this workshop, you will need:
1. **Groq API Key** (For our LLM Brain)
2. **VirusTotal API Key** (For File Hash & Domain checks)
3. **AbuseIPDB API Key** (For IP checks)

Add them to the Google Colab **Secrets (🔑)** tab on the left menu, then run this cell.

In [ ]:
import os
from google.colab import userdata

keys = ["GROQ_API_KEY", "VT_API_KEY", "ABUSEIPDB_KEY"]
for k in keys:
    try:
        os.environ[k] = userdata.get(k)
        print(f"✅ {k} successfully loaded!")
    except userdata.SecretNotFoundError:
        print(f"⚠️ WARNING: {k} missing from Colab Secrets.")
        # Fallback to empty string for testing without errors
        os.environ[k] = "" 

### Testing the AI Agent
Let's make sure our Groq connection is working using the massive 120B model.

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.0
)

print("🤖 Agent says:", llm.invoke("Reply in one short sentence: Are you ready to hunt threats?").content)

--- 
## 🟢 Beginner Profile: Building the Core CTI Scripts
Before we make an autonomous agent, we need to build the raw Python tools that interface with VirusTotal and AbuseIPDB. Run this cell to lay the foundation.

In [ ]:
import requests
import re

VT_KEY = os.environ.get("VT_API_KEY", "")
ABUSE_KEY = os.environ.get("ABUSEIPDB_KEY", "")

def valid_ipv4(ip: str) -> bool:
    parts = ip.split(".")
    return len(parts) == 4 and all(p.isdigit() and 0 <= int(p) <= 255 for p in parts)

def get_ip_reputation(ip_address: str) -> dict:
    """Manual API call to AbuseIPDB"""
    if not valid_ipv4(ip_address):
        return {"status": "error", "reason": "Invalid IP"}
    try:
        resp = requests.get(
            "https://api.abuseipdb.com/api/v2/check",
            headers={"Key": ABUSE_KEY, "Accept": "application/json"},
            params={"ipAddress": ip_address, "maxAgeInDays": 90},
            timeout=10
        ).json().get("data", {})
        return {"ip": ip_address, "abuse_score": resp.get("abuseConfidenceScore", 0)}
    except Exception as e:
        return {"status": "error", "reason": str(e)}

# Let's test it!
print("Testing IP Rep:", get_ip_reputation("8.8.8.8"))

--- 
## 🟡 Intermediate Profile: Exposing Tools via FastMCP
Now we will wrap our threat intelligence code into an official **Model Context Protocol (MCP)** server. This enables any LLM to natively discover and execute our threat intel functions.

In [ ]:
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("CTI Enrichment Server")

@mcp.tool()
def enrich_ip(ip_address: str) -> dict:
    """Check an IPv4 address for threat intelligence using AbuseIPDB.
    Returns abuse confidence score, country, ISP, and total report count."""
    return get_ip_reputation(ip_address)

@mcp.tool()
def enrich_domain(domain: str) -> dict:
    """Check a domain against VirusTotal and return vendor detection count."""
    try:
        attrs = requests.get(
            f"https://www.virustotal.com/api/v3/domains/{domain}",
            headers={"x-apikey": VT_KEY}, timeout=10
        ).json().get("data", {}).get("attributes", {})
        stats = attrs.get("last_analysis_stats", {})
        return {"domain": domain, "malicious": stats.get("malicious", 0)}
    except Exception as e:
        return {"status": "error", "reason": str(e)}

print("✅ FastMCP Server initialized with tools:", [t.name for t in mcp._tool_manager.get_tools()])

--- 
## 🔴 Advanced Profile: The Autonomous Triage Pipeline
For our final challenge, we complete the **Extension Challenge** by writing the WHOIS tool, and we hand the entire toolset to Langchain so our AI Agent can autonomously triage an incoming alert and produce a structured intelligence brief!

In [ ]:
import whois
from langchain.agents import initialize_agent, AgentType
from langchain.tools import tool

# 1. Extension Challenge: The WHOIS Tool
@tool
def get_whois(domain: str) -> str:
    """Gets the WHOIS registration data for a domain to see how old it is."""
    try:
        w = whois.whois(domain)
        return f"Domain: {w.domain_name}, Created: {w.creation_date}, Registrar: {w.registrar}"
    except Exception as e:
        return str(e)

# 2. Langchain Tools (Mirroring our MCP Tools for Colab Execution)
@tool
def ai_enrich_ip(ip: str) -> str:
    """Pass an IP to AbuseIPDB"""
    return str(enrich_ip(ip))

@tool
def ai_enrich_domain(domain: str) -> str:
    """Pass a domain to VirusTotal"""
    return str(enrich_domain(domain))

tools = [ai_enrich_ip, ai_enrich_domain, get_whois]

# 3. Initialize Autonomous Agent
agent = initialize_agent(
    tools, llm, agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

print("🚀 Agent Pipeline Ready!")

In [ ]:
alert_text = """
Alert: Suspicious outbound connection detected.
Source IP: 192.168.10.55 (internal endpoint)
Destination IP: 185.220.101.45
Domain in DNS log: update-secure-patch.net
Timestamp: 2026-03-09 21:43 UTC
"""

prompt = f"""
Triage this alert. You must enrich the Destination IP and the Domain using your tools.
After gathering intelligence, produce a structured brief with:
1. RISK LEVEL: HIGH/MEDIUM/LOW
2. SUMMARY: 2 sentence narrative
3. RECOMMENDED ACTION: Block, Escalate, or Monitor

Raw Alert:
{alert_text}
"""

# Kick off the autonomous loop!
agent.run(prompt)